# Dotreniravanje modela Qwen3-1.7B primjenom QLoRA metode

Ova bilježnica sadrži cjelokupan postupak proveden u praktičnom dijelu završnog rada, od pripreme skupa podataka do dotreniravanja i evaluacije modela.

## Tijek izvođenja

1. Učitavanje i priprema skupa podataka
2. Konfiguracija i priprema modela za QLoRA dotreniravanje
3. Dotreniravanje modela Qwen3-1.7B
4. Spremanje dotreniranog modela
5. Evaluacija dotreniranog modela
6. Evaluacija baznog modela


In [ ]:
!pip install -q datasets sentencepiece safetensors trl

In [ ]:
!pip install -q peft bitsandbytes trl --upgrade

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
     data_files={
        "ISA_240": "../dataset/ISA-240.json",
        "ISA_315": "../dataset/ISA-315.json",
        "ISA_500_draft": "../dataset/ISA-500.json",
        "complex_scenarios": "../dataset/slozeni_scenariji.json"
    }
)

In [ ]:
from datasets import concatenate_datasets
full_datasets = concatenate_datasets([dataset["ISA_240"], dataset["ISA_315"], dataset["ISA_500_draft"], dataset["complex_scenarios"]])

In [ ]:
print(f"Ukupno primjera: {len(full_datasets)}")

split = full_datasets.train_test_split(
    test_size=0.2,
    shuffle=True,
    seed=42
)

test_valid = split["test"].train_test_split(
    test_size=0.5,
    shuffle=True,
    seed=42
)

train_dataset = split["train"]
validation_dataset = test_valid["train"]
test_dataset = test_valid["test"]

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(validation_dataset)}")
print(f"Test: {len(test_dataset)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Specify the pre-trained model name
model_name = "Qwen/Qwen3-1.7B"

# Configure quantization with BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",        # Use NF4 (Normal Float 4) quantization type
    bnb_4bit_compute_dtype=torch.bfloat16, # Set compute dtype to bfloat16 for speed
    bnb_4bit_use_double_quant=True,  # Enable nested quantization for more memory saving
)

# Load the model with quantization configuration
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto", # Automatically distribute model layers across available GPUs/CPU
)

# Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Set padding token if not already set (common requirement for training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Typically right padding for causal LMs

print(f"Model loaded: {model_name}")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
def formatting_func(example):

    messages = [
        {
            "role": "user",
            "content": example["instruction"]
            + ("\n\n" + example["input"] if example["input"].strip() else "")
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [ ]:

# Prepare the model for k-bit training (gradient checkpointing, layer norm scaling)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=32,                         # Rank of the update matrices (higher value = more parameters)
    lora_alpha=64,                # LoRA scaling factor (alpha/r controls the magnitude)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Modules to apply LoRA to (specific to Llama-2 architecture)
    lora_dropout=0.05,            # Dropout probability for LoRA layers
    bias="none",                  # Do not train bias parameters
    task_type="CAUSAL_LM"         # Specify the task type
)

# Apply LoRA configuration to the quantized model
model = get_peft_model(model, lora_config)

# Print the percentage of trainable parameters
model.print_trainable_parameters()

In [ ]:
formatted_dataset = train_dataset.map(
    lambda x: {"text": formatting_func(x)}
)

In [ ]:
import transformers
from trl import SFTTrainer
from transformers import EarlyStoppingCallback

# Configure Training Arguments
from trl import SFTConfig

training_args = SFTConfig(
    output_dir = "../model",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    num_train_epochs=3,

    weight_decay=0.01,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,

    max_length=1024,
    packing=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("Starting QLoRA fine-tuning...")
trainer.train()
print("Training finished.")


In [ ]:
import os

save_path = "../model/final_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model spremljen u:", save_path)
print(os.listdir(save_path))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "../model/final_model"
)

model = PeftModel.from_pretrained(
    base_model,
    "../model/final_model"
)

model.eval()

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm


model.eval()

try:
    model = torch.compile(model)
except:
    pass


test_data = test_dataset


def generate_answer(model, tokenizer, prompt, max_new_tokens=1024):

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    del inputs
    del outputs

    return answer.strip()


SAVE_EVERY = 50

results = []

csv_path = "../results/fine_tuned_results.csv"

os.makedirs(os.path.dirname(csv_path), exist_ok=True)

print()
print("="*60)
print(f"Pokrećem evaluaciju na {len(test_data)} primjera")
print("="*60)


for idx in tqdm(range(len(test_data)), desc="Evaluating"):

    example = test_data[idx]

    try:

        instruction = example["instruction"]
        input_text = example["input"]
        expected_output = example["output"]

        prompt = instruction

        if input_text.strip():
            prompt += "\n\n" + input_text

        ft_answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=1024
        )

        results.append({

            "id": idx,
            "instruction": instruction,
            "input": input_text,
            "question": prompt,
            "expected_output": expected_output,
            "finetuned_model_output": ft_answer

        })

    except Exception as e:

        print(f"\nGreška na primjeru {idx}")
        print(e)

        results.append({

            "id": idx,
            "instruction": "",
            "input": "",
            "question": "",
            "expected_output": "",
            "finetuned_model_output": f"ERROR: {e}"

        })

    if (idx + 1) % SAVE_EVERY == 0 or (idx + 1) == len(test_data):

        pd.DataFrame(results).to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Spremljeno {len(results)}/{len(test_data)}")


df_results = pd.DataFrame(results)

df_results.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("="*60)
print("EVALUACIJA ZAVRŠENA")
print("="*60)
print(f"Rezultati spremljeni u:\n{csv_path}")
print(f"Ukupno primjera: {len(results)}")

In [ ]:
base_model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm


model.eval()

try:
    model = torch.compile(model)
except:
    pass


test_data = test_dataset


def generate_answer(model, tokenizer, prompt, max_new_tokens=1024):

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    del inputs
    del outputs

    return answer.strip()


SAVE_EVERY = 50

results = []

csv_path = "../results/base_model_results.csv"

os.makedirs(os.path.dirname(csv_path), exist_ok=True)

print()
print("="*60)
print(f"Pokrećem evaluaciju na {len(test_data)} primjera")
print("="*60)


for idx in tqdm(range(len(test_data)), desc="Evaluating"):

    example = test_data[idx]

    try:

        instruction = example["instruction"]
        input_text = example["input"]
        expected_output = example["output"]

        prompt = instruction

        if input_text.strip():
            prompt += "\n\n" + input_text

        base_answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=1024
        )

        results.append({

            "id": idx,
            "instruction": instruction,
            "input": input_text,
            "question": prompt,
            "expected_output": expected_output,
            "base_model_output": base_answer

        })

    except Exception as e:

        print(f"\nGreška na primjeru {idx}")
        print(e)

        results.append({

            "id": idx,
            "instruction": "",
            "input": "",
            "question": "",
            "expected_output": "",
            "base_model_output": f"ERROR: {e}"

        })

    if (idx + 1) % SAVE_EVERY == 0 or (idx + 1) == len(test_data):

        pd.DataFrame(results).to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Spremljeno {len(results)}/{len(test_data)}")


df_results = pd.DataFrame(results)

df_results.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("="*60)
print("EVALUACIJA ZAVRŠENA")
print("="*60)
print(f"Rezultati spremljeni u:\n{csv_path}")
print(f"Ukupno primjera: {len(results)}")